# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YoussefZaky208/Flyrank-ML-Track-Assignemnt/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import pandas as pd
import numpy as np
import json
import os

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)


(30000, 44)


## 1. My rule and its reason codes

**Plain words:** a page is worth reviewing for refresh if it's gone stale (not touched in
180+ days) AND it still has real search demand (500+ impressions in the last 90 days). Staleness
alone doesn't mean much if nobody's seeing the page anyway - the demand gate is what makes it
worth someone's time.

Before coding that, I check the two signals it leans on.

**Signal 1 - staleness, behind FlyRank's refresh flags.** Does staleness
(`freshness_tier`) actually line up with decline (`trend_direction == "down"`, used here only to
check the signal - never as a rule input)?

In [ ]:
df["is_declining_label"] = (df["trend_direction"] == "down")

tier_order = ["0-30", "31-90", "91-180", "181+"]
signal1 = df.groupby("freshness_tier")["is_declining_label"].agg(["mean", "count"]).reindex(tier_order)
signal1.columns = ["decline_rate", "n"]
signal1


,decline_rate,n
freshness_tier,,
0-30,0.511377,20480
31-90,0.588571,175
91-180,0.611057,9171
181+,0.471264,174


**Verdict: MIXED.** Decline rate rises from 51% (0-30 days) to 61% (91-180
days), but then *drops* to 47% at 181+ days - the opposite of what "the staler it is, the more
declined it is" would predict. The two extreme tiers are also small (n=175, n=174), so I don't
trust the reversal on its own. Staleness alone is not a clean signal here - good thing I checked
before leaning on it by itself.

**Signal 2 - demand/volume, behind the quick-win logic.** Does higher
search visibility (`impression_tier`) actually mean more click potential, i.e. is "demand" a
real, meaningful axis?

In [ ]:
tier_order2 = ["low", "moderate", "good", "excellent"]
signal2 = df.groupby("impression_tier")["clicks_90d"].agg(["mean", "count"]).reindex(tier_order2)
signal2.columns = ["avg_clicks_90d", "n"]
signal2


,avg_clicks_90d,n
impression_tier,,
low,0.242799,11248
moderate,2.730729,10469
good,30.419709,7205
excellent,215.609462,1078


**Verdict: CONFIRMED.** Average clicks scale hard with impression tier -
0.24 (low) -> 2.73 (moderate) -> 30.4 (good) -> 215.6 (excellent), a clean, monotonic, large jump
at every step across thousands of rows per tier. Demand is real currency here: a page with more
impressions genuinely has more click potential worth protecting or unlocking.

**What this changes about my rule:** since staleness alone is weak (Signal 1) but demand is
strong (Signal 2), I keep both gates - staleness stays as a *requirement*, not the main driver,
and demand (raw impressions) becomes the score itself. That's exactly the transparent-score
pattern from the skill: multiply simple conditions, no fitted weights.

**Reason code:** `stale_visible_page` (single code - matches the one-rule, one-code scope this
week; the multi-reason-code version was W1-W3, this week is deliberately one rule).
**Action label:** `review_for_refresh` if flagged, `monitor` otherwise.

## 2. Build the ranked queue (writes the CSV)

Score = `stale (1/0) * visible (1/0) * impressions_90d` - readable on purpose, no fitted
weights. Everything below comes from features only; `trend_direction`/`trend_pct` (the label)
never enter the score.

In [ ]:
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["score"] = stale * visible * df["impressions_90d"]

df["reason_code"] = np.where(df["score"] > 0, "stale_visible_page", "not_flagged")
df["action"] = np.where(df["score"] > 0, "review_for_refresh", "monitor")

n_flagged = int((df["score"] > 0).sum())
n_clients_flagged = df.loc[df["score"] > 0, "client_id"].nunique()
print(f"Flagged rows: {n_flagged} across {n_clients_flagged} distinct clients (of {df['client_id'].nunique()} total)")

queue = df.sort_values("score", ascending=False)[
    ["content_id", "client_id", "score", "reason_code", "action",
     "impressions_90d", "days_since_last_update", "avg_position", "ctr", "word_count", "content_type"]
]

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote work/outputs/baseline_action_score.csv:", queue.shape)
queue.head(10)


Flagged rows: 17 across 4 distinct clients (of 32 total)


Wrote work/outputs/baseline_action_score.csv: (30000, 11)


,content_id,client_id,score,reason_code,action,impressions_90d,days_since_last_update,avg_position,ctr,word_count,content_type
16751,content_cf56e2e2e282,client_7f2253d7e2,61678,stale_visible_page,review_for_refresh,61678,194,19.7,0.15,5125.0,keyword article
16514,content_7368877ea310,client_7f2253d7e2,59472,stale_visible_page,review_for_refresh,59472,194,24.8,0.13,2591.0,keyword article
7021,content_1bfaa38ff26c,client_7f2253d7e2,25715,stale_visible_page,review_for_refresh,25715,194,22.2,0.23,3861.0,keyword article
21268,content_0a91db491d14,client_7f2253d7e2,13299,stale_visible_page,review_for_refresh,13299,193,10.5,0.49,3478.0,keyword article
11489,content_5feee3994adb,client_7f2253d7e2,7812,stale_visible_page,review_for_refresh,7812,194,39.0,0.01,3590.0,keyword article
12045,content_c2d929d83eaa,client_7f2253d7e2,7558,stale_visible_page,review_for_refresh,7558,193,17.9,0.20,4758.0,keyword article
698,content_b16bd7307b39,client_7f2253d7e2,4590,stale_visible_page,review_for_refresh,4590,194,31.0,0.00,4329.0,keyword article
5327,content_fe16a55cd13d,client_7f2253d7e2,4556,stale_visible_page,review_for_refresh,4556,194,16.4,0.33,3388.0,keyword article
26810,content_ecb6215e79fd,client_7f2253d7e2,4429,stale_visible_page,review_for_refresh,4429,194,25.3,0.38,4486.0,keyword article
20837,content_928af3e22c80,client_7f2253d7e2,1697,stale_visible_page,review_for_refresh,1697,193,15.8,0.12,3118.0,keyword article


**No leakage check:** the score only uses `days_since_last_update` and
`impressions_90d` - both are trailing, already-observed features. `is_declining_label` (from
`trend_direction`) is used two cells below purely to *evaluate* the rule, never inside the score
itself.

In [ ]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

base_rate = float(df["is_declining_label"].mean())
p10 = precision_at_k(df["score"].values, df["is_declining_label"].values, 10)
p_all_flagged = precision_at_k(df["score"].values, df["is_declining_label"].values, n_flagged)

print(f"Base rate (declining, proxy label): {base_rate:.3f}")
print(f"Precision@10: {p10:.3f}")
print(f"Precision@{n_flagged} (all flagged rows): {p_all_flagged:.3f}")

metrics = {
    "rule": "stale (days_since_last_update>=180) AND visible (impressions_90d>=500), score = impressions_90d",
    "reason_code": "stale_visible_page",
    "action_label": "review_for_refresh",
    "n_flagged": n_flagged,
    "n_clients_flagged": int(n_clients_flagged),
    "n_total_rows": int(len(df)),
    "base_rate_proxy_label": round(base_rate, 4),
    "precision_at_10": round(p10, 4),
    "precision_at_all_flagged": round(p_all_flagged, 4),
    "signal_1_verdict": "MIXED",
    "signal_2_verdict": "CONFIRMED",
}
with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Wrote work/outputs/baseline_metrics.json")
metrics


Base rate (declining, proxy label): 0.542
Precision@10: 1.000
Precision@17 (all flagged rows): 0.941
Wrote work/outputs/baseline_metrics.json


{'rule': 'stale (days_since_last_update>=180) AND visible (impressions_90d>=500), score = impressions_90d',
 'reason_code': 'stale_visible_page',
 'action_label': 'review_for_refresh',
 'n_flagged': 17,
 'n_clients_flagged': 4,
 'n_total_rows': 30000,
 'base_rate_proxy_label': 0.5421,
 'precision_at_10': 1.0,
 'precision_at_all_flagged': 0.9412,
 'signal_1_verdict': 'MIXED',
 'signal_2_verdict': 'CONFIRMED'}

## 3. Top-10 review

Precision@10 came back at 1.00 against the proxy label - which sounds great, but with only 17
flagged rows total and a small n, I'm reading this with a skeptic's eye below, not celebrating
it.

In [ ]:
top10 = queue.head(10).reset_index(drop=True)

for i, row in top10.iterrows():
    ctr_note = "a very low CTR for its position" if row["ctr"] < 0.2 else "a CTR that looks okay for now"
    wrong_if = (
        f"wrong if this {row['days_since_last_update']:.0f}-day 'staleness' is just a CMS "
        f"timestamp quirk (e.g. templated content that doesn't re-save on edit) rather than "
        f"actually neglected content -- or if, despite {ctr_note}, the page recovers on its "
        f"own next month with no edit at all"
    )
    print(f"{i+1}. {row['content_id']} (client {row['client_id']})")
    print(f"   action: {row['action']}  |  reason: {row['reason_code']}")
    print(f"   why: {row['impressions_90d']:.0f} impressions/90d, stale {row['days_since_last_update']:.0f} days, "
          f"position {row['avg_position']:.1f}, ctr {row['ctr']:.2f}%")
    print(f"   what would make it wrong: {wrong_if}")
    print()


1. content_cf56e2e2e282 (client client_7f2253d7e2)
   action: review_for_refresh  |  reason: stale_visible_page
   why: 61678 impressions/90d, stale 194 days, position 19.7, ctr 0.15%
   what would make it wrong: wrong if this 194-day 'staleness' is just a CMS timestamp quirk (e.g. templated content that doesn't re-save on edit) rather than actually neglected content -- or if, despite a very low CTR for its position, the page recovers on its own next month with no edit at all

2. content_7368877ea310 (client client_7f2253d7e2)
   action: review_for_refresh  |  reason: stale_visible_page
   why: 59472 impressions/90d, stale 194 days, position 24.8, ctr 0.13%
   what would make it wrong: wrong if this 194-day 'staleness' is just a CMS timestamp quirk (e.g. templated content that doesn't re-save on edit) rather than actually neglected content -- or if, despite a very low CTR for its position, the page recovers on its own next month with no edit at all

3. content_1bfaa38ff26c (client clie

## 4. Weak picks + leakage check

**The real weak spot: client concentration.** All 10 of my top-10 rows come from a *single*
client (`client_7f2253d7e2`) - checked below. Across all 17 flagged rows, 12 of 17 are still
that same client. That's the honest weak pick here, not any one row: the AND-combination of
strict staleness (180+ days) and strict visibility (500+ impressions) happens to be rare enough
that it collapses onto whichever client's content ops process produces both traits together.
**What would make this whole list wrong:** if that client simply updates their CMS timestamps
differently than everyone else (so "180 days since update" doesn't mean the same thing for them
as for other clients), my rule isn't finding genuine neglect - it's finding one client's
metadata quirk. I'd want to check that before trusting this queue across the full client base.

**Leakage check:** `trend_direction` / `trend_pct` were used only in `is_declining_label`, which
appears **nowhere** in the score, reason code, or action - confirmed by re-reading the score
formula above. No future-window data exists in this starter slice at all (everything is a
trailing-90-day snapshot), so there's no forward-looking window to leak from here.

**Product-flag check:** the guide is clear that FlyRank's real product flags
(`health_score`, `needs_ctr_fix`, `is_quick_win`, `priority_score`, `action_type`) are never
shipped in this dataset - confirmed below by listing all 44 starter-CSV columns and checking
none of those names appear. Nothing to accidentally leak because it was never in the data to
begin with.

In [ ]:
# Confirm the client-concentration finding with real numbers
top10_clients = top10["client_id"].value_counts()
all_flagged_clients = queue[queue["score"] > 0]["client_id"].value_counts()

print("Client distribution in top 10:")
print(top10_clients)
print()
print("Client distribution across all 17 flagged rows:")
print(all_flagged_clients)
print()

# Confirm no label-derived column made it into the score inputs
score_inputs = {"days_since_last_update", "impressions_90d"}
label_cols = {"trend_direction", "trend_pct", "is_declining_label"}
print("Score built only from:", score_inputs)
print("Overlap with label-derived columns (should be empty set):", score_inputs & label_cols)

# Confirm no FlyRank product-decision flags exist in this dataset to leak from
product_flag_names = {"health_score", "needs_ctr_fix", "is_quick_win", "priority_score", "action_type"}
print("Product-flag columns present in starter CSV (should be empty set):", product_flag_names & set(df.columns))


Client distribution in top 10:
client_id
client_7f2253d7e2    10
Name: count, dtype: int64

Client distribution across all 17 flagged rows:
client_id
client_7f2253d7e2    12
client_d029fa3a95     3
client_9400f1b21c     1
client_4ec9599fc2     1
Name: count, dtype: int64

Score built only from: {'days_since_last_update', 'impressions_90d'}
Overlap with label-derived columns (should be empty set): set()
Product-flag columns present in starter CSV (should be empty set): set()


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere - only pseudonymized IDs
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.